# Bootstrap Validation — RF Ad-hoc Pipeline (nb32)

B=25 bootstrap resamples of the nb32 pipeline.

| Step | Description |
|:----:|-------------|
| 1 | Ad-hoc genre consolidation — keep 8 genres, fold rest into `other` |
| 2 | Grid search (18 combinations, `n_estimators=500` fixed) |
| 3 | Centrality ablation (2³ = 8 subsets) with winning params |
| 4 | Final fit + OOF threshold tuning (maximize F1) |

**Tracked per iteration:** test AUC · logloss · brier · OOF threshold · precision · recall · F1

In [1]:
import warnings
warnings.filterwarnings('ignore')

import itertools
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import gaussian_kde

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score, brier_score_loss, log_loss,
    precision_score, recall_score, f1_score
)
from sklearn.model_selection import train_test_split, StratifiedKFold

In [ ]:
B            = 25
RANDOM_STATE = 42
N_SPLITS     = 5
LAM          = 0.3
N_EST        = 500   # fixed from nb32 grid winner

KEEP_GENRES = [
    'R&B/Soul/Funk', 'Hip Hop/Rap', 'Latin', 'Electronic/Dance',
    'Folk', 'Pop', 'Country/Americana', 'Rock',
]

CENTRALITY_FEATS = [
    'betweenness_centrality_top20_rolling5',
    'eigenvector_centrality_top20_rolling5',
    'harmonic_closeness_centrality_top20_rolling5',
]

# 18 combinations — n_estimators fixed
PARAM_GRID = [
    {'n_estimators': N_EST, 'max_depth': d, 'min_samples_leaf': l, 'max_features': f,
     'class_weight': 'balanced', 'random_state': RANDOM_STATE, 'n_jobs': -1}
    for d, l, f in itertools.product([3, 5, 8], [1, 5, 10], ['sqrt', 0.5])
]

# nb32 single-run reference
NB32_AUC       = 0.7662
NB32_GAP       = 0.0320
NB29_RF_MEAN   = 0.7602

df = pd.read_csv('../df_artists_final.csv', index_col=0).reset_index()
X  = df.drop(columns=['top_20_hitmaker'])
y  = df['top_20_hitmaker']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

print(f'Train: {X_train.shape}  Test: {X_test.shape}')
print(f'Grid size: {len(PARAM_GRID)} combinations  (n_estimators fixed at {N_EST})')

In [ ]:
def consolidate_genres(X, keep_genres):
    X = X.copy()
    keep_cols      = [f'artist_genre_{g}' for g in keep_genres]
    all_genre_cols = [c for c in X.columns if c.startswith('artist_genre_')]
    drop_cols      = [c for c in all_genre_cols if c not in keep_cols]
    X['artist_genre_other'] = X[drop_cols].max(axis=1)
    X = X.drop(columns=drop_cols)
    return X


def cv_evaluate(X, y, params, skf):
    cv_aucs, tr_aucs = [], []
    for tr, va in skf.split(X, y):
        m = RandomForestClassifier(**params)
        m.fit(X.iloc[tr], y.iloc[tr])
        cv_aucs.append(roc_auc_score(y.iloc[va], m.predict_proba(X.iloc[va])[:, 1]))
        tr_aucs.append(roc_auc_score(y.iloc[tr], m.predict_proba(X.iloc[tr])[:, 1]))
    cv  = np.mean(cv_aucs)
    gap = np.mean(tr_aucs) - cv
    return {'CV AUC': cv, 'Gap': gap, 'Penalized': cv - LAM * gap}


def run_pipeline(X_tr, y_tr, X_te, y_te, seed):
    skf_b = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)

    # Impute
    imp      = SimpleImputer(strategy='median')
    X_tr_imp = pd.DataFrame(imp.fit_transform(X_tr), columns=X_tr.columns)
    X_te_imp = pd.DataFrame(imp.transform(X_te),     columns=X_te.columns)

    # Step 1 — genre consolidation
    X_tr_c = consolidate_genres(X_tr_imp, KEEP_GENRES)
    X_te_c = consolidate_genres(X_te_imp, KEEP_GENRES)

    # Step 2 — grid search
    best_params, best_score = None, -np.inf
    for params in PARAM_GRID:
        res = cv_evaluate(X_tr_c, y_tr, params, skf_b)
        if res['Penalized'] > best_score:
            best_score, best_params = res['Penalized'], params

    # Step 3 — centrality ablation
    non_centrality = [c for c in X_tr_c.columns if c not in CENTRALITY_FEATS]
    best_subset, best_cv = None, -np.inf
    for k in range(len(CENTRALITY_FEATS) + 1):
        for combo in itertools.combinations(CENTRALITY_FEATS, k):
            feats = non_centrality + list(combo)
            res   = cv_evaluate(X_tr_c[feats], y_tr, best_params, skf_b)
            if res['CV AUC'] > best_cv:
                best_cv, best_subset = res['CV AUC'], feats

    X_tr_f = X_tr_c[best_subset]
    X_te_f = X_te_c[best_subset]

    # Step 4 — OOF threshold tuning (maximize F1)
    oof_proba = np.zeros(len(y_tr))
    for tr, va in skf_b.split(X_tr_f, y_tr):
        m = RandomForestClassifier(**best_params)
        m.fit(X_tr_f.iloc[tr], y_tr.iloc[tr])
        oof_proba[va] = m.predict_proba(X_tr_f.iloc[va])[:, 1]

    thresholds = np.linspace(0.05, 0.95, 181)
    best_t, best_f1 = 0.5, 0.0
    for t in thresholds:
        f1 = f1_score(y_tr, (oof_proba >= t).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t

    # Final fit
    model = RandomForestClassifier(**best_params)
    model.fit(X_tr_f, y_tr)
    test_proba = model.predict_proba(X_te_f)[:, 1]
    y_pred     = (test_proba >= best_t).astype(int)

    return {
        'test_auc':   roc_auc_score(y_te, test_proba),
        'logloss':    log_loss(y_te, test_proba),
        'brier':      brier_score_loss(y_te, test_proba),
        'threshold':  best_t,
        'precision':  precision_score(y_te, y_pred, zero_division=0),
        'recall':     recall_score(y_te, y_pred, zero_division=0),
        'f1':         f1_score(y_te, y_pred, zero_division=0),
        'n_features': len(best_subset),
    }

## Bootstrap Loop

In [ ]:
METRICS = ['test_auc', 'logloss', 'brier', 'threshold', 'precision', 'recall', 'f1', 'n_features']
results = {'RF': {m: [] for m in METRICS}, 'Baseline': {'test_auc': []}}

rng = np.random.default_rng(RANDOM_STATE)

for b in range(B):
    seed     = RANDOM_STATE + b
    boot_idx = rng.choice(len(X_train), size=len(X_train), replace=True)
    X_tr_b   = X_train.iloc[boot_idx].reset_index(drop=True)
    y_tr_b   = y_train.iloc[boot_idx].reset_index(drop=True)

    print(f'[{b+1:02d}/{B}] RF... ', end='', flush=True)
    try:
        res = run_pipeline(X_tr_b, y_tr_b, X_test, y_test, seed)
        for m in METRICS:
            results['RF'][m].append(res[m])
        print(f'AUC={res["test_auc"]:.3f}  thr={res["threshold"]:.2f}  '
              f'P={res["precision"]:.3f}  R={res["recall"]:.3f}  F1={res["f1"]:.3f}  '
              f'n={res["n_features"]} | ', end='', flush=True)
    except Exception as e:
        print(f'ERROR({e}) | ', end='', flush=True)
        for m in METRICS:
            results['RF'][m].append(np.nan)

    dummy = DummyClassifier(strategy='stratified', random_state=seed)
    dummy.fit(X_tr_b, y_tr_b)
    results['Baseline']['test_auc'].append(
        roc_auc_score(y_test, dummy.predict_proba(X_test)[:, 1])
    )
    print(f'Baseline={results["Baseline"]["test_auc"][-1]:.3f}')

    with open('bootstrap_adhoc_results.pkl', 'wb') as f:
        pickle.dump(results, f)

print('\nDone. Results saved to bootstrap_adhoc_results.pkl')

## Results

In [ ]:
with open('bootstrap_adhoc_results.pkl', 'rb') as f:
    results = pickle.load(f)

palette = {'RF': '#55A868', 'Baseline': '#aaaaaa'}
rf_vals  = [v for v in results['RF']['test_auc'] if not np.isnan(v)]
bl_vals  = [v for v in results['Baseline']['test_auc'] if not np.isnan(v)]
bl_mean  = np.mean(bl_vals)

plot_metrics = [
    ('test_auc',  'Test AUC',   (0.55, 0.90)),
    ('threshold', 'OOF Threshold', (0.20, 0.80)),
    ('f1',        'F1 (OOF thr)', (0.40, 0.90)),
    ('precision', 'Precision',  (0.30, 1.00)),
    ('recall',    'Recall',     (0.30, 1.00)),
    ('brier',     'Brier',      (0.10, 0.40)),
]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle(
    f'Bootstrap Validation — RF Ad-hoc Pipeline  (B={B})\n'
    'Ad-hoc genre consolidation · Grid search (18 combos) · Centrality ablation',
    fontsize=13, fontweight='bold'
)

for ax, (metric, label, xlim) in zip(axes.flat, plot_metrics):
    vals  = [v for v in results['RF'][metric] if not np.isnan(v)]
    if not vals:
        ax.set_title(f'{label}\n(no data)'); continue

    color = palette['RF']
    mu    = np.mean(vals)
    p5    = np.percentile(vals, 5)
    p95   = np.percentile(vals, 95)
    x_grid = np.linspace(*xlim, 300)

    ax.hist(vals, bins=10, color=color, alpha=0.35, density=True)
    try:
        kde   = gaussian_kde(vals, bw_method='scott')
        y_kde = kde(x_grid)
        ax.plot(x_grid, y_kde, color=color, lw=2.0)
        mask = (x_grid >= p5) & (x_grid <= p95)
        ax.fill_between(x_grid[mask], y_kde[mask], alpha=0.30, color=color,
                        label=f'90% CI [{p5:.3f}, {p95:.3f}]')
    except Exception:
        pass

    ax.axvline(mu, color=color, lw=2.0, linestyle='--', label=f'mean={mu:.3f}')

    # Reference lines for AUC panel
    if metric == 'test_auc':
        ax.axvline(NB32_AUC,     color='black',  lw=1.5, linestyle=':',  label=f'nb32 single={NB32_AUC:.3f}')
        ax.axvline(NB29_RF_MEAN, color='orange', lw=1.5, linestyle='--', label=f'nb29 mean={NB29_RF_MEAN:.3f}')
        ax.axvline(bl_mean,      color='grey',   lw=1.0, linestyle=':',  label=f'baseline={bl_mean:.3f}')

    # Reference line for threshold panel
    if metric == 'threshold':
        ax.axvline(0.46, color='black', lw=1.5, linestyle=':', label='nb32 thr=0.46')

    ax.set_xlabel(label, fontsize=9)
    ax.set_ylabel('Density', fontsize=9)
    ax.set_title(label, fontweight='bold')
    ax.set_xlim(xlim)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('bootstrap_adhoc_plot.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
with open('bootstrap_adhoc_results.pkl', 'rb') as f:
    results = pickle.load(f)

print(f'{"Metric":<15}  {"Mean":>8}  {"Std":>8}  {"90% CI":>22}')
print('─' * 60)

for metric, label, _ in plot_metrics:
    vals = [v for v in results['RF'][metric] if not np.isnan(v)]
    if not vals: continue
    ci = f'[{np.percentile(vals, 5):.3f}, {np.percentile(vals, 95):.3f}]'
    print(f'{label:<15}  {np.mean(vals):>8.4f}  {np.std(vals):>8.4f}  {ci:>22}')

print()
bl_aucs = [v for v in results['Baseline']['test_auc'] if not np.isnan(v)]
rf_aucs = [v for v in results['RF']['test_auc']       if not np.isnan(v)]
print(f'Baseline mean AUC : {np.mean(bl_aucs):.4f}')
print(f'RF vs baseline    : {np.mean(rf_aucs) - np.mean(bl_aucs):+.4f}')
print(f'nb32 single-run   : {NB32_AUC:.4f}')
print(f'nb29 fixed-param  : {NB29_RF_MEAN:.4f}')